# 02 - Scoring Validation

This notebook validates the full weighted scoring pipeline (`api.scoring.score_candidate`) end-to-end against the sample data, and checks that the weighted score formula behaves as expected under different weight configurations.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from pathlib import Path
import pandas as pd
from api.parsing import normalize_text, load_skills_taxonomy
from api.scoring import compute_tfidf_similarity, score_candidate, DEFAULT_WEIGHTS

DATA_DIR = Path('..') / 'data'
taxonomy = load_skills_taxonomy()
jd_text = normalize_text((DATA_DIR / 'sample_jds' / 'data_analyst.txt').read_text(encoding='utf-8'))
resume_files = sorted((DATA_DIR / 'sample_resumes').glob('*.txt'))
resume_texts = [normalize_text(f.read_text(encoding='utf-8')) for f in resume_files]
resume_names = [f.name for f in resume_files]

In [2]:
sims = compute_tfidf_similarity(jd_text, resume_texts)

rows = []
for name, text, sim in zip(resume_names, resume_texts, sims):
    result = score_candidate(jd_text, text, sim, taxonomy=taxonomy)
    result['filename'] = name
    rows.append(result)

df = pd.DataFrame(rows).sort_values('overall_score', ascending=False).reset_index(drop=True)
df[['filename', 'overall_score', 'similarity', 'skill_overlap', 'experience_years',
    'experience_fit', 'education_level', 'education_fit']]

,filename,overall_score,similarity,skill_overlap,experience_years,experience_fit,education_level,education_fit
0,priya_sharma_data_analyst_senior.txt,65.54,0.4385,0.60,8.02,1.00,3,1.00
1,maria_gonzalez_data_analyst_mid.txt,54.28,0.3070,0.40,6.27,1.00,2,1.00
2,daniel_osei_data_analyst_junior.txt,51.30,0.3575,0.36,2.43,0.81,2,1.00
3,sophia_lindqvist_data_engineer.txt,50.42,0.2704,0.32,6.44,1.00,2,1.00
4,aisha_rahman_ml_engineer.txt,42.21,0.1251,0.24,6.94,1.00,3,1.00
5,james_okafor_backend_senior.txt,34.06,0.0416,0.08,7.95,1.00,3,1.00
6,carlos_mendez_sales_weak_fit.txt,34.04,0.0409,0.08,8.52,1.00,2,1.00
7,lena_kowalski_backend_mid.txt,33.23,0.0508,0.04,6.02,1.00,2,1.00
8,tom_becker_backend_junior.txt,31.59,0.0647,0.08,3.43,1.00,1,0.66


## Sanity checks

- The senior/mid data-analyst resumes (Priya, Maria) should rank above the clearly-mismatched sales resume (Carlos) and pure backend engineers.
- Experience years extracted should roughly match what's written in each resume (hand-verified against the .txt files in `data/sample_resumes/`).
- `overall_score` must always fall within [0, 100].

In [3]:
assert df['overall_score'].between(0, 100).all(), 'scores out of bounds!'
assert df['skill_overlap'].between(0, 1).all()
assert df['experience_fit'].between(0, 1).all()
assert df['education_fit'].between(0, 1).all()
print('All bounds checks passed.')

All bounds checks passed.


In [4]:
top_row = df.iloc[0]
print(top_row['filename'])
assert 'data_analyst' in top_row['filename'] or 'data_engineer' in top_row['filename'], (
    'Expected a data-analyst-flavored resume to rank first for the Data Analyst JD'
)
print('Top-ranked resume matches expectation:', top_row['filename'])

priya_sharma_data_analyst_senior.txt
Top-ranked resume matches expectation: priya_sharma_data_analyst_senior.txt


## Weight sensitivity check

Re-score with skills-only weighting (1.0 on skills, 0 elsewhere) and confirm the ranking shifts to prioritize raw skill overlap over text similarity.

In [5]:
skills_only_weights = {'similarity': 0.0, 'skills': 1.0, 'experience': 0.0, 'education': 0.0}

rows2 = []
for name, text, sim in zip(resume_names, resume_texts, sims):
    result = score_candidate(jd_text, text, sim, weights=skills_only_weights, taxonomy=taxonomy)
    result['filename'] = name
    rows2.append(result)

df2 = pd.DataFrame(rows2).sort_values('overall_score', ascending=False).reset_index(drop=True)
df2[['filename', 'overall_score', 'skill_overlap']]

,filename,overall_score,skill_overlap
0,priya_sharma_data_analyst_senior.txt,60.0,0.60
1,maria_gonzalez_data_analyst_mid.txt,40.0,0.40
2,daniel_osei_data_analyst_junior.txt,36.0,0.36
3,sophia_lindqvist_data_engineer.txt,32.0,0.32
4,aisha_rahman_ml_engineer.txt,24.0,0.24
5,carlos_mendez_sales_weak_fit.txt,8.0,0.08
6,james_okafor_backend_senior.txt,8.0,0.08
7,tom_becker_backend_junior.txt,8.0,0.08
8,lena_kowalski_backend_mid.txt,4.0,0.04


In [6]:
# Confirm overall_score under skills-only weighting is exactly skill_overlap * 100
import numpy as np
expected = (df2['skill_overlap'] * 100).round(2)
assert np.allclose(df2['overall_score'], expected, atol=0.01), 'weighted score math mismatch'
print('Weighted score formula validated under skills-only weighting.')

Weighted score formula validated under skills-only weighting.
